# DDE alternans, spiral fragmentation and dynamic restitution
Run the three code cells in order in a fresh Colab CPU runtime. This notebook is self-contained: the first cell writes the full Python/C++ source included as plain text, then compiles the scalar kernel. No Drive mount, private URL, or ZIP upload is needed for this notebook. The code ZIP provides the same sources separately for convenient editing.

All original algebraic gates are retained, including both maxima in h. The initial entire voltage history is the new resting equilibrium. Cell pacing: I=5.6 uA/cm² for 0.75 ms; BCL 10/7.7 ms; 800 beats. Tissue: 0.75-cm square, D=0.001 cm²/ms, S1 at 0–0.5 ms, S2 at 7–7.5 ms, I=30 uA/cm²; no later current. Block 2 defaults to the audited Heun/nodal boundary implementation, so exact trajectories can differ from the earlier Euler snapshots. Main tissue arrays use approximately 2.7 GB; computation takes several minutes on a CPU, depending on Colab allocation.

Block 3 starts at BCL 12 ms and reduces BCL by 0.01 ms down to 7 ms, retaining full history. Each stage uses 800–6400 beats, with a convergence test over the last 40 cycles. Both final APDs are retained separately. Missing captures remain missing; gray crosses mark stages that did not settle to captured period one/two. The shorter APD is plotted against **its own preceding DI**, not the smaller of two DIs chosen independently. The derivative is evaluated only inside contiguous, measured increasing APD–DI segments, without monotonic regression or interpolation across unresolved gaps. Panel (c) magnifies the raw short-APD branch, including its small turn in DI. Panel (d) shows slopes on contiguous increasing segments. The optional local scan (`LOCAL_REFINE=True` by default) is independently equilibrated at BCL 7.7 ms and then continues with a finer 0.001-ms BCL step and dt=0.0005 ms.

This is dynamic restitution, not S1–S2 restitution. In this DDE regime the sweep has irregular/capture-failure intervals as well as clean alternans, so it is not a textbook single period-doubling split. A slope above one on a selected dynamic branch does not establish a memoryless restitution map or the mechanism of tissue breakup. Numerical maxima can be sensitive to BCL resolution, especially near a turn in DI.

Figures are saved in `dde_colab/figs_chap3` as PNG and PDF; raw events, settings and branch metrics are saved in `dde_colab/data`. Download that directory or selected files using Colab's file browser after computation.


In [ ]:
# Block 1: all solver sources are embedded below as plain text.
from pathlib import Path
import sys, subprocess, importlib.util
for package in ['numpy', 'scipy', 'matplotlib']:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
ROOT = Path.cwd() / 'dde_colab'
(ROOT / 'code').mkdir(parents=True, exist_ok=True)
SOURCES = {
'common.py': r'''from pathlib import Path
import ctypes as ct, hashlib, subprocess, os
import numpy as np
ROOT=Path(__file__).resolve().parent.parent
DATA=ROOT/'data'; FIG=ROOT/'figs_chap3'
DATA.mkdir(exist_ok=True); FIG.mkdir(exist_ok=True)
PARAMS=np.array([1.,-82.,45.,-78.3,-18.,.152,.003,55.,53.,.115,7.2,1.11])
BLUE='#2166ac';GOLD='#d18b12';RED='#b2182b'

def style():
 import matplotlib.pyplot as plt
 plt.rcParams.update({'font.family':'DejaVu Sans','font.size':9,'axes.spines.top':False,'axes.spines.right':False,'pdf.fonttype':42,'axes.linewidth':.65})

def compile_lib(name,openmp=False):
 src=ROOT/'code'/name;so=src.with_name('_'+src.stem+'_'+hashlib.sha256(src.read_bytes()).hexdigest()[:12]+'.so')
 if not so.exists():
  tmp=so.with_name(so.name+f'.{os.getpid()}.tmp')
  cmd=['g++','-O3','-std=c++17','-fPIC','-shared']
  if openmp:cmd+=['-fopenmp']
  subprocess.run(cmd+[str(src),'-o',str(tmp)],check=True);os.replace(tmp,so)
 return ct.CDLL(str(so))

class PacedCell:
 def __init__(self,dt=.001,method='heun'):
  if method not in ['euler','heun']:raise ValueError('Unknown method')
  self.dt=dt;self.heun=int(method=='heun');self.lib=compile_lib('continuation_core.cpp')
  dbl=np.ctypeslib.ndpointer(dtype=np.float64,flags='C_CONTIGUOUS')
  self.lib.new_cell.argtypes=[dbl,ct.c_double];self.lib.new_cell.restype=ct.c_void_p
  self.lib.delete_cell.argtypes=[ct.c_void_p]
  self.lib.paced_block.argtypes=[ct.c_void_p,ct.c_double,ct.c_double,ct.c_int,ct.c_double,ct.c_double,ct.c_int,dbl]
  self.ptr=self.lib.new_cell(PARAMS,self.dt)
 def pace(self,bcl,beats=800,amplitude=5.6,duration=.75):
  for x in [bcl,duration]:
   if abs(x/self.dt-round(x/self.dt))>1e-7:raise ValueError('Pulse boundaries must lie on the time grid')
  out=np.empty((beats,5));ret=self.lib.paced_block(self.ptr,self.dt,bcl,beats,amplitude,duration,self.heun,out)
  if ret:raise FloatingPointError(f'Pacing failed: {ret}')
  return out
 def close(self):
  if self.ptr:self.lib.delete_cell(self.ptr);self.ptr=None
 def __del__(self):
  if getattr(self,'ptr',None):self.close()

def savefig(fig,name):
 for ext in ['png','pdf']:fig.savefig(FIG/f'{name}.{ext}',dpi=300,bbox_inches='tight')
 return FIG/f'{name}.png'
''',
'delayed_model.py': r'''"""Delayed AP model with the exact Chapter 2 mathematical gates.

No additional dynamical variables or ionic currents are introduced.
The shared C++ kernel compiles automatically with a local C++17 compiler.
"""
from dataclasses import dataclass, asdict, replace
from pathlib import Path
import ctypes as ct
import subprocess
import hashlib
import numpy as np

ROOT = Path(__file__).resolve().parent

@dataclass(frozen=True)
class Parameters:
    cm: float = 1.
    vk: float = -82.
    vna: float = 45.
    vth: float = -60.
    vc: float = -18.
    gkb: float = .152
    gnab: float = .016
    ck: float = 13.125
    cna: float = 36.96
    tm: float = .19903
    th: float = 2.75768
    tk: float = 1.3
    cap_h: bool = True

    @property
    def rest(self):
        return (self.gkb*self.vk+self.gnab*self.vna)/(self.gkb+self.gnab)

    def vector(self):
        if not self.cap_h:raise ValueError('The Chapter 2 max-max h gate is required')
        return np.array(list(asdict(self).values()), dtype=np.float64)

def _library():
    src=ROOT/'delayed_core.cpp'
    digest=hashlib.sha256(src.read_bytes()).hexdigest()[:12]
    lib=ROOT/f'_delayed_core_{digest}.so'
    if not lib.exists() or lib.stat().st_mtime < src.stat().st_mtime:
        subprocess.run(['g++','-O3','-std=c++17','-fPIC','-shared',str(src),'-o',str(lib)],check=True)
    dll = ct.CDLL(str(lib))
    dbl = np.ctypeslib.ndpointer(dtype=np.float64,flags='C_CONTIGUOUS')
    integer = np.ctypeslib.ndpointer(dtype=np.int32,flags='C_CONTIGUOUS')
    dll.cell_trace.argtypes=[dbl,ct.c_double,ct.c_int,dbl,ct.c_int,ct.c_double,ct.c_double,ct.c_double,dbl]
    return dll

_DLL = None
def library():
    global _DLL
    if _DLL is None: _DLL=_library()
    return _DLL

def trace(p=Parameters(), starts=(10.,), end=40., dt=.002, amplitude=30.,duration=.5,initial=None):
    if dt<=0 or dt>min(p.tm,p.th,p.tk): raise ValueError('Require 0 < dt <= minimum delay')
    starts=np.ascontiguousarray(starts,dtype=np.float64)
    if np.any(np.diff(starts)<duration-1e-9):raise ValueError('Pulses must be sorted and nonoverlapping')
    n=int(round(end/dt))+1
    v=np.empty(n)
    done=library().cell_trace(p.vector(),dt,n,starts,len(starts),duration,amplitude,p.rest if initial is None else initial,v)
    if done<n:raise FloatingPointError(f'Invalid voltage at time {done*dt:g}')
    return np.arange(n)*dt,v
''',
'delayed_core.cpp': r'''#include <algorithm>
#include <cmath>
#include <vector>
#include <cstdint>

// Parameter order is declared in delayed_model.py. Gates are normalized.
struct Model {
    double cm,vk,vna,vth,vc,gkb,gnab,ck,cna,tm,th,tk,rest;
    Model(const double* p):cm(p[0]),vk(p[1]),vna(p[2]),vth(p[3]),vc(p[4]),
        gkb(p[5]),gnab(p[6]),ck(p[7]),cna(p[8]),tm(p[9]),th(p[10]),tk(p[11]),
        rest((gkb*vk+gnab*vna)/(gkb+gnab)) {}
    inline double rhs(double v,double vm,double vh,double vn,double stim) const {
        double m=std::max(vm-vth,0.0)/(vna-vth);
        double n=std::max(vn-vth,0.0)/(vna-vth);
        double h=std::max(vc-std::max(vh,vth),0.0)/(vc-vk);
        return ((gkb+ck*n)*(vk-v)+(gnab+cna*m*h)*(vna-v)+stim)/cm;
    }
};
struct Delay {
    int k; double f;
    Delay(double tau,double dt):k((int)std::floor(tau/dt)),f(tau/dt-k){}
    inline double get(const std::vector<double>& h,int pos)const {
        int b=(pos-k+(int)h.size())%(int)h.size();
        int a=(b-1+(int)h.size())%(int)h.size();
        return (1-f)*h[b]+f*h[a];
    }
};

extern "C" {
// Arbitrary pulse train. Euler with linear interpolation of delayed voltages.
int cell_trace(const double* p,double dt,int nt,const double* starts,int ns,
               double duration,double amplitude,double initial,double* trace) {
    Model m(p); int nh=(int)std::ceil(std::max({m.tm,m.th,m.tk})/dt)+3;
    std::vector<double> hist(nh,initial);
    Delay dm(m.tm,dt),dh(m.th,dt),dk(m.tk,dt);
    double v=initial; int s=0,pos=0;
    for(int i=0;i<nt;i++) {
        hist[pos]=v;trace[i]=v;
        if(!std::isfinite(v)||v < -200||v>200)return i+1;
        if(i+1==nt)break;
        double t=i*dt;
        while(s<ns&&t>=starts[s]+duration-1e-10)s++;
        double stim=(s<ns&&t>=starts[s]-1e-10)?amplitude:0;
        v+=dt*m.rhs(v,dm.get(hist,pos),dh.get(hist,pos),dk.get(hist,pos),stim);
        pos=(pos+1)%nh;
    }
    return nt;
}


}
''',
'appendix_config.py': r'''"""Clean single-cell alternans and breakup, with the original double-max h gate."""
from delayed_model import Parameters
REFERENCE=Parameters()
SELECTED=Parameters(vth=-78.3,gnab=.003,ck=55.,cna=53.,tm=.115,tk=1.11,th=7.2)
REGULAR_BCL=10.
ALTERNANS_BCL=7.7
CELL_AMPLITUDE=5.6
CELL_DURATION=.75
TISSUE_AMPLITUDE=30.
TISSUE_DURATION=.5
''',
'plot_pacing_original.py': r'''"""Publication figures from raw voltage; no trace filtering or smoothing."""
from pathlib import Path
import argparse,numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from appendix_config import REGULAR_BCL,ALTERNANS_BCL,CELL_DURATION
ROOT=Path(__file__).resolve().parent.parent;D=ROOT/'data';F=ROOT/'figs_chap3'
BLUE='#2166ac';GOLD='#d18b12';RED='#b2182b'
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':8.5,'axes.labelsize':8.5,'axes.titlesize':9.,'xtick.labelsize':7.5,'ytick.labelsize':7.5,
 'axes.spines.top':False,'axes.spines.right':False,'pdf.fonttype':42,'axes.linewidth':.65})
def save(fig,name,formats):
 for ext in formats:fig.savefig(F/f'{name}.{ext}',dpi=350,bbox_inches='tight',pad_inches=.045)
 plt.close(fig)
def arrow(ax,x1,x2,y,label):
 ax.annotate('',xy=(x1,y),xytext=(x2,y),arrowprops={'arrowstyle':'<->','color':RED,'lw':.8,'shrinkA':0,'shrinkB':0})
 ax.text((x1+x2)/2,y+4,label,ha='center',va='bottom',color=RED,fontsize=7.5)
def pacing(formats):
 fig,axs=plt.subplots(2,2,figsize=(7.5,4.2),gridspec_kw={'width_ratios':[3.6,1.65]},layout='constrained')
 for row,(bcl,label) in enumerate([(REGULAR_BCL,'Regular pacing'),(ALTERNANS_BCL,'APD alternans')]):
  z=np.load(D/f'pacing_bcl{bcl:g}.npz');t=z['t'];v=z['V'];starts=z['starts'];a=z['activation'];r=z['repolarization'];apd=z['apd0'];xmax=40.
  ax=axs[row,0];mask=t<=xmax+1e-8;ax.plot(t[mask],v[mask],color=BLUE,lw=1.05)
  for at,rt,duration in zip(a,r,apd):
   if rt>xmax:continue
   q=(t>=at)&(t<=rt);color=BLUE if not row or duration<apd.mean() else GOLD
   ax.fill_between(t[q],0,v[q],color=color,alpha=.33,linewidth=0)
  ax.axhline(0,color='.45',ls=(0,(3,2)),lw=.65,zorder=0);ax.text(xmax-.2,3,'0 mV',ha='right',va='bottom',color='.4',fontsize=7)
  for st in starts[starts<xmax]:ax.plot([st,st+CELL_DURATION],[-84,-84],color='.25',lw=2.2,solid_capstyle='butt')
  arrow(ax,r[0],a[1],-30,fr'$\mathrm{{DI}}_0={a[1]-r[0]:.3f}$ ms')
  if row:arrow(ax,r[1],a[2],-30,fr'$\mathrm{{DI}}_0={a[2]-r[1]:.3f}$ ms')
  ax.set(xlim=(-.2,xmax+.2),ylim=(-88,58),ylabel='$V$ (mV)',xticks=np.arange(0,41,5),yticks=[-80,-40,0,40],xlabel='Time (ms)' if row else '')
  ax.set_title(f'({chr(97+row)}) {label}, BCL = {bcl:g} ms',loc='left',pad=6)
  aa=axs[row,1];order=[int(np.argmin(apd)),int(np.argmax(apd))] if row else [0]
  for k,j in enumerate(order):
   mask=(t>=starts[j]-1e-8)&(t<starts[j]+bcl-1e-8);tt=t[mask]-a[j];vv=v[mask];color=BLUE if k==0 else GOLD
   aa.plot(tt,vv,color=color,lw=1.35,label=fr'$\mathrm{{APD}}_0={apd[j]:.3f}$ ms');q=(tt>=0)&(tt<=apd[j]);aa.fill_between(tt[q],0,vv[q],color=color,alpha=.22,lw=0)
  aa.axhline(0,color='.45',ls=(0,(3,2)),lw=.65);aa.axvline(0,color='.65',ls=':',lw=.65)
  aa.set(xlim=(-1.,2.1),ylim=(-88,58),yticks=[-80,-40,0,40],xticks=[-1,0,1,2],xlabel=r'$t-t_{\mathrm{act}}$ (ms)' if row else '')
  aa.legend(loc='lower left',bbox_to_anchor=(-.01,1.015),frameon=False,fontsize=7.,handlelength=1.4,borderaxespad=0.);aa.tick_params(labelleft=False)
 fig.suptitle('Single-cell action potentials',fontsize=9.3);save(fig,'dde_clean_cell_alternans',formats)
''',
'figure1.py': r'''"""Figure 1: exactly the original scalar Euler protocol and plot style."""
import numpy as np,json,time
from common import ROOT,DATA,FIG,PARAMS
from delayed_model import Parameters,trace
from plot_pacing_original import pacing
from dataclasses import asdict
DT=.00025;BEATS=800;AMPLITUDE=5.6;DURATION=.75
P=Parameters(*PARAMS,True)

def crossings(t,v):
 i=np.flatnonzero((v[:-1]<0)&(v[1:]>=0));j=np.flatnonzero((v[:-1]>=0)&(v[1:]<0))
 a=t[i]-v[i]*(t[i+1]-t[i])/(v[i+1]-v[i]);r=t[j]-v[j]*(t[j+1]-t[j])/(v[j+1]-v[j])
 if len(a)!=len(r) or np.any(r<=a):raise ValueError('Incomplete or unpaired APs')
 return a,r

def main():
 metrics=[]
 for bcl in [10.,7.7]:
  starts=bcl*np.arange(BEATS);t,v=trace(P,starts,BEATS*bcl,DT,AMPLITUDE,DURATION)
  a,r=crossings(t,v)
  if len(a)!=BEATS:raise ValueError('Expected one AP per stimulus')
  apd=r-a;di=a[1:]-r[:-1];lo=starts[-6];keep=t>=lo-1e-9
  np.savez_compressed(DATA/f'pacing_bcl{bcl:g}.npz',t=t[keep]-lo,V=v[keep],starts=starts[-6:]-lo,activation=a[-6:]-lo,repolarization=r[-6:]-lo,apd0=apd[-6:],preceding_di0=di[-6:])
  rec=dict(bcl_ms=bcl,last_two_apd_ms=apd[-2:].tolist(),last_two_di_ms=di[-2:].tolist(),dt_ms=DT,beats=BEATS)
  print(rec,flush=True);metrics.append(rec);del t,v
 pacing(['pdf','png'])
 for ext in ['pdf','png']:
  old=FIG/f'dde_clean_cell_alternans.{ext}';old.replace(FIG/f'dde_regular_alternans.{ext}')
 (DATA/'figure1_metrics.json').write_text(json.dumps(dict(parameters=asdict(P),runs=metrics),indent=2))
 print('Saved',FIG/'dde_regular_alternans.pdf')
 return FIG/'dde_regular_alternans.png'
if __name__=='__main__':main()
''',
'independent_solver.cpp': r'''// Independent implementation for the numerical audit. Does not include the
// published Model, Delay, or spatial solver. Same double-max gate equations.
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <vector>
#include <omp.h>
struct Params {
 double cm,vk,vna,vt,vc,gk,gn,ck,cn,tm,th,tk;
 Params(const double* p):cm(p[0]),vk(p[1]),vna(p[2]),vt(p[3]),vc(p[4]),gk(p[5]),gn(p[6]),ck(p[7]),cn(p[8]),tm(p[9]),th(p[10]),tk(p[11]){}
};
static inline double ionic(const Params& p,double v,double vm,double vh,double vn,double current) {
 double sodium_activation=std::max(vm-p.vt,0.)/(p.vna-p.vt);
 double potassium_activation=std::max(vn-p.vt,0.)/(p.vna-p.vt);
 double availability=std::max(p.vc-std::max(vh,p.vt),0.)/(p.vc-p.vk);
 double ik=(p.gk+p.ck*potassium_activation)*(p.vk-v);
 double ina=(p.gn+p.cn*sodium_activation*availability)*(p.vna-v);
 return (ik+ina+current)/p.cm;
}
extern "C" int independent_pde(const double* parameters,double dt,int nt,int nx,double length,double diffusion,
 int method,int boundary,int nthreads,int stride,float* frames,float* delayed,double* probes,double* stats) {
 Params p(parameters);int np=nx*nx,nh=(int)std::ceil(std::max({p.tm,p.tk,p.th,1.})/dt)+3;
 double rest=(p.gk*p.vk+p.gn*p.vna)/(p.gk+p.gn),dx=length/(nx-1),fac=diffusion/(dx*dx);
 if(dt>std::min({p.tm,p.th,p.tk})||4*fac*dt>1.)return -999001;
 std::vector<double> history((size_t)nh*np,rest),v(np,rest),pred(np),k1(np);
 double tau[4]={p.tm,p.th,p.tk,.5},frac[4];int k[4],pos=0,failed=0;
 for(int j=0;j<4;j++){k[j]=(int)std::floor(tau[j]/dt);frac[j]=tau[j]/dt-k[j];}
 int ip[16],jj=0;
 for(double y:{.15,.30,.45,.60})for(double x:{.15,.30,.45,.60})ip[jj++]=(int)std::llround(y/dx)*nx+(int)std::llround(x/dx);
 omp_set_num_threads(nthreads);
 #pragma omp parallel
 {
  for(int it=0;it<=nt;it++) {
   if(it%stride==0) {
    int out=it/stride;const double* h0=history.data()+(size_t)((pos-k[3]+nh)%nh)*np;
    const double* h1=history.data()+(size_t)((pos-k[3]-1+nh)%nh)*np;
    #pragma omp for schedule(static)
    for(int q=0;q<np;q++) {
     frames[(size_t)out*np+q]=(float)v[q];delayed[(size_t)out*np+q]=(float)((1-frac[3])*h0[q]+frac[3]*h1[q]);
    }
    #pragma omp single
    {
     double lo=1e99,hi=-1e99,active=0;
     for(double w:v){lo=std::min(lo,w);hi=std::max(hi,w);active+=w>-20.;if(!std::isfinite(w)||w<-200||w>200)failed=it+1;}
     stats[3*out]=active/np;stats[3*out+1]=lo;stats[3*out+2]=hi;
    }
   }
   #pragma omp single
   {for(int j=0;j<16;j++)probes[(size_t)it*16+j]=v[ip[j]];}
   if(it==nt||failed)break;
   double* current=history.data()+(size_t)pos*np;
   #pragma omp for schedule(static)
   for(int q=0;q<np;q++)current[q]=v[q];
   const double *a[3],*b[3];
   for(int j=0;j<3;j++){a[j]=history.data()+(size_t)((pos-k[j]+nh)%nh)*np;b[j]=history.data()+(size_t)((pos-k[j]-1+nh)%nh)*np;}
   double t=it*dt;bool s1=t<.5-1e-10,s2=t>=7.-1e-10&&t<7.5-1e-10;
   #pragma omp for schedule(static)
   for(int y=0;y<nx;y++)for(int x=0;x<nx;x++) {
    int q=y*nx+x;double vm=(1-frac[0])*a[0][q]+frac[0]*b[0][q];
    double vh=(1-frac[1])*a[1][q]+frac[1]*b[1][q];double vn=(1-frac[2])*a[2][q]+frac[2]*b[2][q];
    double current=((s1&&x*dx<=.05)||(s2&&y*dx<=.5*length))?30.:0.;
    double reaction=ionic(p,v[q],vm,vh,vn,current);
    double lap=v[y*nx+std::max(x-1,0)]+v[y*nx+std::min(x+1,nx-1)]+v[std::max(y-1,0)*nx+x]+v[std::min(y+1,nx-1)*nx+x]-4*v[q];
    if(boundary==1) {
     if(x==0)lap+=v[q+1]-v[q];if(x==nx-1)lap+=v[q-1]-v[q];
     if(y==0)lap+=v[q+nx]-v[q];if(y==nx-1)lap+=v[q-nx]-v[q];
    }
    k1[q]=reaction+fac*lap;
    // Preserve the reference Euler operator order for the comparison.
    if(method==0){pred[q]=v[q]+dt*reaction;pred[q]+=dt*fac*lap;}
    else pred[q]=v[q]+dt*k1[q];
   }
   if(method==1) {
    // At the second stage use the known history at t+dt-tau, never the
    // current predictor as delayed history. tau >= dt in every audit run.
    for(int j=0;j<3;j++){a[j]=history.data()+(size_t)((pos-k[j]+1+nh)%nh)*np;b[j]=history.data()+(size_t)((pos-k[j]+nh)%nh)*np;}
    #pragma omp for schedule(static)
    for(int y=0;y<nx;y++)for(int x=0;x<nx;x++) {
     int q=y*nx+x;double vm=(1-frac[0])*a[0][q]+frac[0]*b[0][q];double vh=(1-frac[1])*a[1][q]+frac[1]*b[1][q];double vn=(1-frac[2])*a[2][q]+frac[2]*b[2][q];
     // Pulses are piecewise constant and every discontinuity is on a step
     // boundary. Use the same one-sided forcing throughout this step.
     double current=((s1&&x*dx<=.05)||(s2&&y*dx<=.5*length))?30.:0.;
     double lap=pred[y*nx+std::max(x-1,0)]+pred[y*nx+std::min(x+1,nx-1)]+pred[std::max(y-1,0)*nx+x]+pred[std::min(y+1,nx-1)*nx+x]-4*pred[q];
     if(boundary==1) {
      if(x==0)lap+=pred[q+1]-pred[q];if(x==nx-1)lap+=pred[q-1]-pred[q];
      if(y==0)lap+=pred[q+nx]-pred[q];if(y==nx-1)lap+=pred[q-nx]-pred[q];
     }
     double k2=ionic(p,pred[q],vm,vh,vn,current)+fac*lap;v[q]+=.5*dt*(k1[q]+k2);
    }
   }
   #pragma omp single
   {
    if(method==0)v.swap(pred);pos=(pos+1)%nh;
    if(it>0&&it%(int)std::llround(10./dt)==0)std::fprintf(stderr,"independent %s nx=%d t=%.2f ms\n",method?"Heun":"Euler",nx,it*dt);
   }
  }
 }
 return failed?-failed:nt/stride+1;
}
''',
'figure2.py': r'''"""Figure 2: same gates/parameters; audited tissue implementation.

Default: Heun, double-precision full history, consistent nodal Neumann BC.
For the earlier Euler snapshots, use method='euler', boundary='legacy'.
This preserves the historical boundary convention for comparison only.
"""
from common import *
import time,json
import matplotlib.pyplot as plt

NX=201;DT=.001;END=500.;SAMPLE=.5
METHOD='heun';BOUNDARY='nodal';THREADS=2
SNAPSHOTS=[14.,20.,22.5,23.5,26.5,27.5,100.,500.]


def run(nx=NX,dt=DT,end=END,sample=SAMPLE,method=METHOD,boundary=BOUNDARY,threads=THREADS):
 if method not in ['heun','euler'] or boundary not in ['nodal','legacy']:raise ValueError('Unknown solver option')
 nt=round(end/dt);stride=round(sample/dt);nf=nt//stride+1
 if abs(nt*dt-end)>1e-9 or abs(stride*dt-sample)>1e-9:raise ValueError('Time settings must align with time steps')
 # Full delay history is float64 (~2.33 GB on the default grid).
 # Saved voltage fields are float32; output precision does not enter evolution.
 gb=(8*(np.ceil(7.2/dt)+3)*nx*nx+8*nf*nx*nx+8*(nt+1)*16)/1e9
 print(f'Estimated main arrays: {gb:.2f} GB; running to {end:g} ms.',flush=True)
 dll=compile_lib('independent_solver.cpp',openmp=True)
 dbl=np.ctypeslib.ndpointer(dtype=np.float64,flags='C_CONTIGUOUS');flt=np.ctypeslib.ndpointer(dtype=np.float32,flags='C_CONTIGUOUS')
 dll.independent_pde.argtypes=[dbl,ct.c_double,ct.c_int,ct.c_int,ct.c_double,ct.c_double,ct.c_int,ct.c_int,ct.c_int,ct.c_int,flt,flt,dbl,dbl]
 V=np.empty((nf,nx,nx),np.float32);lag=np.empty_like(V);probes=np.empty((nt+1,16));stats=np.empty((nf,3))
 tic=time.time();n=dll.independent_pde(PARAMS,dt,nt,nx,.75,.001,int(method=='heun'),int(boundary=='nodal'),threads,stride,V,lag,probes,stats)
 if n<0:raise FloatingPointError(f'Tissue solver returned {n}')
 t=np.arange(n)*stride*dt;chosen=np.array([x for x in SNAPSHOTS if x<=t[-1]+1e-9])
 ids=np.array([np.argmin(abs(t-x)) for x in chosen])
 if len(ids)==0:raise ValueError('No requested snapshot falls in the simulated interval')
 meta=dict(parameters=PARAMS.tolist(),scheme=method,boundary=boundary,history_dtype='float64',nx=nx,dt_ms=dt,end_ms=end,
  length_cm=.75,diffusion_cm2_per_ms=.001,s1_ms=[0.,.5],s2_ms=[7.,7.5],amplitude_uA_per_cm2=30.,seconds=time.time()-tic)
 np.savez_compressed(DATA/'breakup_snapshots.npz',t=t[ids],V=V[ids],meta_json=json.dumps(meta))
 # Native-time probe traces permit threshold timing without coarse frame aliasing.
 np.savez_compressed(DATA/'breakup_probes.npz',t=np.arange(nt+1)*dt,V=probes)
 (DATA/'figure2_settings.json').write_text(json.dumps(meta,indent=2))
 print('Tissue run finished:',meta,flush=True)
 return t[ids],V[ids]


def plot(t,V):
 style();ncol=4;nrow=int(np.ceil(len(t)/ncol));L=.75
 fig=plt.figure(figsize=(7.3,2.05*nrow));gs=fig.add_gridspec(nrow,5,width_ratios=[1,1,1,1,.075],wspace=.38,hspace=.43,left=.066,right=.94,bottom=.115,top=.94)
 for j,(tt,v) in enumerate(zip(t,V)):
  ax=fig.add_subplot(gs[j//4,j%4]);im=ax.imshow(v,origin='lower',extent=[0,L,0,L],cmap='RdBu_r',vmin=-82,vmax=45,interpolation='nearest')
  ax.set_title(f'$t={tt:g}$ ms',fontsize=9,pad=4);ax.set(xticks=[0,.25,.5,.75],yticks=[0,.25,.5,.75],xlabel='$x$ (cm)',ylabel='$y$ (cm)' if j%4==0 else '')
  if j%4:ax.tick_params(labelleft=False)
  ax.tick_params(labelsize=6.8,pad=2,length=2)
  for spine in ax.spines.values():spine.set_visible(True);spine.set_linewidth(.5)
 cb=fig.colorbar(im,cax=fig.add_subplot(gs[:,4]),ticks=[-80,-60,-40,-20,0,20,40]);cb.set_label('$V$ (mV)')
 path=savefig(fig,'dde_spiral_breakup');plt.close(fig);return path


def main(**kwargs):
 t,V=run(**kwargs);return plot(t,V)
if __name__=='__main__':main()
''',
'continuation_core.cpp': r'''// Continuous-history pacing: identical max-max h gate. Euler or Heun.
#include <algorithm>
#include <cmath>
#include <vector>
struct Cell {
 double cm,vk,vna,vt,vc,gk,gn,ck,cn,tm,th,tk;
 int nh,pos=0; double v,time=0,at=NAN,last_rep=NAN,active_di=NAN;
 std::vector<double> h; int k[3];double f[3];
 Cell(const double*p,double dt):cm(p[0]),vk(p[1]),vna(p[2]),vt(p[3]),vc(p[4]),gk(p[5]),gn(p[6]),ck(p[7]),cn(p[8]),tm(p[9]),th(p[10]),tk(p[11]) {
  v=(gk*vk+gn*vna)/(gk+gn);nh=(int)std::ceil(std::max({tm,th,tk})/dt)+4;h.assign(nh,v);
  double tau[3]={tm,th,tk};for(int j=0;j<3;j++){k[j]=(int)std::floor(tau[j]/dt);f[j]=tau[j]/dt-k[j];}
 }
 double delayed(int j,int stage){return (1-f[j])*h[(pos-k[j]+stage+nh)%nh]+f[j]*h[(pos-k[j]+stage-1+nh)%nh];}
 double rhs(double w,int stage,double I){
  double m=std::max(delayed(0,stage)-vt,0.)/(vna-vt);
  double hh=std::max(vc-std::max(delayed(1,stage),vt),0.)/(vc-vk);
  double n=std::max(delayed(2,stage)-vt,0.)/(vna-vt);
  return ((gk+ck*n)*(vk-w)+(gn+cn*m*hh)*(vna-w)+I)/cm;
 }
};
extern "C" {
 void* new_cell(const double*p,double dt){return new Cell(p,dt);}
 void delete_cell(void*x){delete (Cell*)x;}
 // cols: APD, preceding DI, latency, upcrossings, downcrossings.
 // Only accept one complete up/down event per stimulus cycle. Missing=NaN.
 int paced_block(void*x,double dt,double bcl,int beats,double amp,double duration,int heun,double*out){
  Cell&c=*(Cell*)x;int nc=(int)std::llround(bcl/dt),nd=(int)std::llround(duration/dt);
  if(dt>=std::min({c.tm,c.th,c.tk})||nc<=nd)return -1;
  for(int b=0;b<beats;b++){
   double* row=out+b*5;row[0]=row[1]=row[2]=NAN;int nu=0,nr=0;bool paired=false;
   double start=c.time;
   for(int j=0;j<nc;j++){
    c.h[c.pos]=c.v;double I=j<nd?amp:0.,k1=c.rhs(c.v,0,I);
    double w=heun?c.v+.5*dt*(k1+c.rhs(c.v+dt*k1,1,I)):c.v+dt*k1;
    if(!std::isfinite(w)||w < -200||w>200)return -2;
    if(c.v<0&&w>=0){nu++;c.at=c.time+dt*(-c.v)/(w-c.v);c.active_di=c.at-c.last_rep;row[2]=c.at-start;}
    if(c.v>=0&&w<0){nr++;double rt=c.time+dt*(-c.v)/(w-c.v);
     if(std::isfinite(c.at)&&c.at>=start-1e-8){row[0]=rt-c.at;row[1]=c.active_di;paired=true;}
     c.last_rep=rt;c.at=NAN;
    }
    c.v=w;c.pos=(c.pos+1)%c.nh;c.time+=dt;
   }
   row[3]=nu;row[4]=nr;
   if(nu!=1||nr!=1||!paired)row[0]=row[1]=row[2]=NAN;
  }
  return 0;
 }
}
''',
'figure3.py': r'''"""Slow dynamic restitution: continuous history, 0-mV crossing events.

This is a pacing-dependent bifurcation diagram and dynamic restitution,
not S1-S2 restitution. A shorter-APD branch is selected with its OWN DI.
No monotonic regression and no slope threshold are imposed on the data.
"""
import json,time
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator
from common import *

# User controls (all in ms). Continue from 12 ms downwards without resetting.
START_BCL=12.;END_BCL=7.;BCL_STEP=.01
DT=.001;METHOD='heun'
MIN_BEATS=800;MAX_BEATS=6400;BATCH=200;WINDOW=40
PERIOD_TOL=1e-5   # both APD and preceding DI, over the last 40 beats
AMPLITUDE=5.6;DURATION=.75
LOCAL_REFINE=True  # Additional branch check at dt=.0005 and dBCL=.001


def convergence(tail):
 # P1/P2 require 1:1 capture, one complete AP and finite preceding DI.
 z=tail[-WINDOW:]
 if len(z)<WINDOW or not np.isfinite(z[:,:3]).all() or not np.all(z[:,3:]==1):
  return False,'capture failure',np.nan
 e1=float(np.max(np.abs(z[1:,:2]-z[:-1,:2])))
 e2=float(np.max(np.abs(z[2:,:2]-z[:-2,:2])))
 if e1<PERIOD_TOL:return True,'period 1',e1
 if e2<PERIOD_TOL:return True,'period 2',e2
 return False,'unresolved',e2


def sweep(dt=DT,start=START_BCL,end=END_BCL,step=BCL_STEP):
 bcls=np.round(np.linspace(start,end,round((start-end)/step)+1),8)
 cell=PacedCell(dt,METHOD);records=[];tails=[];tic=time.time()
 try:
  for i,bcl in enumerate(bcls):
   used=0;tail=np.empty((0,5));stable=False;status='unresolved';error=np.nan
   while used<MAX_BEATS:
    z=cell.pace(float(bcl),BATCH,AMPLITUDE,DURATION);used+=BATCH
    tail=np.concatenate([tail,z])[-WINDOW:]
    if used>=MIN_BEATS:
     stable,status,error=convergence(tail)
     if stable:break
     # Do not silently turn failed capture into an APD=0 branch.
     if used>=MIN_BEATS and not np.isfinite(tail[-20:,0]).any():break
     if used>=MIN_BEATS and np.all(tail[-20:,3]==0):break
   rec=dict(bcl_ms=float(bcl),beats=used,settled=stable,status=status,
            periodicity_error_ms=float(error),apd_last2_ms=tail[-2:,0].tolist(),
            di_last2_ms=tail[-2:,1].tolist(),latency_last2_ms=tail[-2:,2].tolist(),
            upcrossings_last2=tail[-2:,3].tolist(),downcrossings_last2=tail[-2:,4].tolist())
   records.append(rec);tails.append(tail.copy())
   if i%10==0 or i==len(bcls)-1:
    print(f'BCL {bcl:.2f}: {status}; APD {tail[-2:,0]}; beats {used}; elapsed {time.time()-tic:.0f} s',flush=True)
    # Checkpoint measurements; the on-going cell still retains full history.
    (DATA/f'dynamic_restitution_dt{dt:g}_progress.json').write_text(json.dumps(records,indent=2))
 finally:cell.close()
 np.savez_compressed(DATA/f'dynamic_restitution_dt{dt:g}.npz',bcl=bcls,last40=np.array(tails))
 report=dict(protocol='descending BCL; continuous full voltage history; dynamic restitution',
   dt_ms=dt,method=METHOD,parameters=PARAMS.tolist(),amplitude=AMPLITUDE,duration_ms=DURATION,
   bcl_start_ms=start,bcl_end_ms=end,bcl_step_ms=step,min_beats=MIN_BEATS,max_beats=MAX_BEATS,
   period_tolerance_ms=PERIOD_TOL,runs=records,elapsed_seconds=time.time()-tic)
 (DATA/f'dynamic_restitution_dt{dt:g}.json').write_text(json.dumps(report,indent=2))
 return report


def branch_segments(di,apd,bcl,valid):
 # Split in original pacing order at failures, reversals, or capture gaps.
 # Never sort across a turning point or connect different branches.
 pieces=[];current=[]
 for i in range(len(bcl)):
  if not valid[i] or not np.isfinite(di[i]+apd[i]):
   if len(current)>=3:pieces.append(current)
   current=[];continue
  if current:
   prev=current[-1];dx=di[i]-di[prev];dy=apd[i]-apd[prev]
   # descending BCL should give descending DI on the retained branch.
   # A positive restitution segment has decreasing APD in the same order.
   if dx>=-1e-7 or dy>0:
    if len(current)>=3:pieces.append(current)
    current=[]
  current.append(i)
 if len(current)>=3:pieces.append(current)
 return [np.array(x) for x in pieces]


def plot(report,local=None):
 style();runs=report['runs'];bcl=np.array([x['bcl_ms'] for x in runs])
 a=np.array([x['apd_last2_ms'] for x in runs]);d=np.array([x['di_last2_ms'] for x in runs])
 ok=np.array([x['settled'] for x in runs]);sel=np.argmin(np.where(np.isfinite(a),a,np.inf),axis=1)
 short=a[np.arange(len(a)),sel];own_di=d[np.arange(len(a)),sel]
 fig,axes=plt.subplots(2,2,figsize=(9.0,6.5),layout='constrained');ax0,ax1,ax2,ax3=axes.ravel()
 for j,c in [(0,BLUE),(1,GOLD)]:
  ax0.scatter(bcl[ok],a[ok,j],s=9,c=c,alpha=.8,label=f'Final AP {j+1}',edgecolors='none')
 if (~ok).any():
  ax0.scatter(np.repeat(bcl[~ok],2),a[~ok].ravel(),marker='x',s=12,c='.6',label='Unsettled / failed capture')
 ax0.set(xlabel='BCL (ms)',ylabel=r'$\mathrm{APD}_0$ (ms)',title='(a) Descending-BCL bifurcation diagram')
 ax0.invert_xaxis();ax0.legend(frameon=False,fontsize=7.5)
 ax1.scatter(d[~ok].ravel(),a[~ok].ravel(),marker='x',s=8,c='.8',alpha=.45,label='Unsettled / failed capture')
 ax1.scatter(d[ok].ravel(),a[ok].ravel(),s=7,c='.65',alpha=.45,label='Both final APs',edgecolors='none')
 ax1.scatter(own_di[ok],short[ok],s=10,c=BLUE,label='Shorter APD, paired preceding DI',edgecolors='none')
 ax1.set(xlabel=r'Preceding $\mathrm{DI}_0$ (ms)',ylabel=r'$\mathrm{APD}_0$ (ms)',title='(b) Dynamic restitution')
 ax1.legend(frameon=False,fontsize=7,loc='best')
 all_metrics=[];zoom=None
 def draw_segments(b,dd,aa,valid,color,label):
  pieces=branch_segments(dd,aa,b,valid);metrics=[];largest=-np.inf;which=None
  for ip,ids in enumerate(pieces):
   ii=ids[::-1];x=dd[ii];y=aa[ii];p=PchipInterpolator(x,y)
   xx=np.linspace(x[0],x[-1],max(100,4*len(x)));yy=p(xx);ss=p.derivative()(xx)
   ax1.plot(xx,yy,color=color,lw=.9)
   ax2.plot(xx,yy,color=color,lw=1)
   ax3.plot(xx,ss,color=color,lw=1.2,label=label if ip==0 else None)
   candidates=list(x)
   for kk in range(len(x)-1):
    cubic=p.c[0,kk];quadratic=p.c[1,kk]
    if abs(cubic)>1e-30:
     vertex=-quadratic/(3*cubic)
     if 0<vertex<x[kk+1]-x[kk]:candidates.append(float(x[kk]+vertex))
   candidates=np.array(candidates);derivatives=p.derivative()(candidates);k=int(np.argmax(derivatives))
   segment_max=float(derivatives[k]);secants=np.diff(y)/np.diff(x)
   record=dict(bcl_range_ms=[float(b[ids[0]]),float(b[ids[-1]])],di_range_ms=[float(x[0]),float(x[-1])],points=len(ids),max_pchip_slope=segment_max,max_adjacent_secant_slope=float(secants.max()))
   metrics.append(record)
   if segment_max>largest:largest=segment_max;which=record
  return dict(label=label,max_pchip_slope=largest,segments=metrics),which
 base,steep=draw_segments(bcl,own_di,short,ok,BLUE,r'$\Delta\mathrm{BCL}=0.01$ ms')
 all_metrics.append(base);zoom=steep
 # All measured short-APD points remain visible, including nonmonotonic parts.
 ax2.plot(own_di[ok],short[ok],'.',ms=3,color=BLUE,alpha=.7,label='Full sweep')
 local_info=None
 if local is not None:
  lb,tail=local;la=tail[:,-2:,0];ld=tail[:,-2:,1]
  lok=np.array([convergence(z)[0] for z in tail]);jj=np.argmin(np.where(np.isfinite(la),la,np.inf),axis=1)
  ls=la[np.arange(len(la)),jj];lds=ld[np.arange(len(la)),jj]
  # Connect only consecutive settled samples in continuation order. The line
  # may turn back in DI; do not sort it into a false single-valued function.
  ax2.plot(np.where(lok,lds,np.nan),np.where(lok,ls,np.nan),'o-',ms=2.3,lw=.7,color=GOLD,label='Local finer scan')
  detail,local_steep=draw_segments(lb,lds,ls,lok,GOLD,r'Local $\Delta\mathrm{BCL}=0.001$ ms')
  all_metrics.append(detail)
  if local_steep is not None:zoom=local_steep
  local_info=dict(start_bcl_ms=7.7,end_bcl_ms=7.385,step_ms=.001,dt_ms=.0005,
    initialization='1600 beats at BCL=7.7 from rest, then continuation on the clean period-two branch',
    settled_points=int(lok.sum()),points=len(lb))
 if zoom is not None:
  lo,hi=zoom['di_range_ms'];width=hi-lo
  # Zoom near the steep end, and include neighbouring raw data to show folds.
  ax2.set_xlim(lo-max(.15*width,.001),min(hi,lo+max(.25*width,.012)))
  ax2.set_ylim(.889,.906)
  ax3.set_xlim(lo-max(.03*width,.0002),hi+max(.03*width,.0002))
 ax2.set(xlabel=r'Preceding $\mathrm{DI}_0$ (ms)',ylabel=r'$\mathrm{APD}_0$ (ms)',title='(c) Detail of the shorter-APD branch')
 ax2.ticklabel_format(useOffset=False,axis='x');ax2.legend(frameon=False,fontsize=7.5)
 ax3.axhline(1,color=RED,lw=1,ls='--',label='Slope = 1')
 ax3.set(xlabel=r'Preceding $\mathrm{DI}_0$ (ms)',ylabel=r'$d\mathrm{APD}_0/d\mathrm{DI}_0$',title='(d) Slopes within increasing segments')
 ax3.ticklabel_format(useOffset=False,axis='x');ax3.legend(frameon=False,fontsize=7.5,loc='upper right')
 alternating=ok&(np.abs(a[:,0]-a[:,1])>1e-3)
 metrics=dict(curves=all_metrics,local_check=local_info,
   largest_sampled_alternating_bcl_ms=float(bcl[alternating].max()) if alternating.any() else None,
   unsettled_or_failed_bcls_ms=bcl[~ok].tolist(),
   interpretation='Protocol- and branch-dependent dynamic slope. Near a DI turning point the reported maximum depends on sampling. Not an S1-S2 curve or proof of the breakup mechanism.')
 (DATA/'restitution_branch_metrics.json').write_text(json.dumps(metrics,indent=2))
 for curve in all_metrics:
  print(curve['label'],'maximum fitted slope:',curve['max_pchip_slope'],flush=True)
 path=savefig(fig,'dde_dynamic_restitution');plt.close(fig);return path


def main():
 report=sweep(dt=DT,start=START_BCL,end=END_BCL,step=BCL_STEP)
 local=None
 if LOCAL_REFINE:
  from check_local_slope import main as local_sweep
  local=local_sweep(dt=.0005)
 return plot(report,local=local)
if __name__=='__main__':main()
''',
'check_local_slope.py': r'''"""Local branch check, independently equilibrated at the clean P2 BCL.

This checks the same settled shorter-APD branch near 7.4 ms at smaller
BCL steps, not the full 12-to-7 ms protocol or its irregular transitions.
"""
from common import *
from figure3 import convergence
import json,time

def main(dt=.0005):
 c=PacedCell(dt);c.pace(7.7,1600);bcls=np.round(np.arange(7.7,7.3849,-.001),3);tails=[];tic=time.time()
 try:
  for i,b in enumerate(bcls):
   z=c.pace(float(b),800);ok,status,error=convergence(z)
   used=800
   while not ok and used<6400:
    z=c.pace(float(b),200);used+=200;ok,status,error=convergence(z)
   tails.append(z[-40:])
   if i%25==0:print('Local check',dt,b,status,z[-2:,:2],time.time()-tic,flush=True)
 finally:c.close()
 tails=np.array(tails);np.savez_compressed(DATA/f'local_slope_dt{dt:g}.npz',bcl=bcls,last40=tails)
 vals=tails[:,-2:,0];dis=tails[:,-2:,1];jj=np.argmin(vals,axis=1);a=vals[np.arange(len(vals)),jj];d=dis[np.arange(len(vals)),jj]
 for j in range(len(bcls)-15,len(bcls)):
  print('End',bcls[j],a[j],d[j],(a[j]-a[j-1])/(d[j]-d[j-1]),flush=True)
 return bcls,tails
if __name__=='__main__':main()
''',
}
for name, text in SOURCES.items():
    (ROOT / 'code' / name).write_text(text)
sys.path.insert(0, str(ROOT / 'code'))
from IPython.display import display, Image
import figure1
display(Image(filename=str(figure1.main())))
print('PDF:', ROOT / 'figs_chap3' / 'dde_regular_alternans.pdf')


## Tissue snapshots
Use the full 500-ms interval for the persistent-activity figure. No history is reset during stimulation or integration.

In [ ]:
# Block 2: same ionic model; audited Heun solver and nodal Neumann boundaries.
import figure2
# Use method='euler', boundary='legacy' for the earlier Euler snapshots.
display(Image(filename=str(figure2.main(
    method='heun', boundary='nodal', nx=201, dt=0.001,
    end=500.0, threads=2))))
print('PDF:', ROOT / 'figs_chap3' / 'dde_spiral_breakup.pdf')


## Slow continuation and dynamic restitution
The scan can spend longer in irregular regimes. Reaching the beat limit without the specified convergence is flagged, not accepted as a periodic branch.

In [ ]:
# Block 3: carry the full delay history from one BCL to the next.
import figure3
figure3.START_BCL = 12.0
figure3.END_BCL = 7.0
figure3.BCL_STEP = 0.01
figure3.DT = 0.001
figure3.MIN_BEATS = 800
figure3.MAX_BEATS = 6400
figure3.LOCAL_REFINE = True
display(Image(filename=str(figure3.main())))
print('PDF:', ROOT / 'figs_chap3' / 'dde_dynamic_restitution.pdf')
